In [3]:
print("Hello World")

Hello World


In [9]:
# IT3011 - Theory & Practices in Statistical Modelling
# Group Name: In_Sighted
# Group Assignment: Recognition for Good Work Increases Employee Engagement
# Dataset: IBM HR Analytics Employee Attrition & Performance (Kaggle)
# =============================================================================
 
# ---- Impoerting Libraries ----

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind, f_oneway, pearsonr


from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
 
import warnings
warnings.filterwarnings("ignore")

In [10]:
# Plotting style 

In [23]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f9f9f9",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.size": 11,
})
PALETTE = "Set2"

In [11]:
# SECTION 1 — Load & Inspect Data
# The dataset was downloaded from Kaggle (IBM HR Analytics Attrition dataset).
# It contains 1,470 employee records and 35 variables covering demographics,
# job characteristics, compensation, and satisfaction scores.

In [24]:
df = pd.read_csv("C:\\Users\\Mahesh Masinghe\\Desktop\\TPSM Project\\TPSM_HR_Dataset.csv")

In [25]:
print("=" * 60)
print("SECTION 1 — Dataset Overview")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nAttrition breakdown:\n{df['Attrition'].value_counts()}")
print(f"\nMissing values: {df.isnull().sum().sum()} (none expected in this dataset)")

SECTION 1 — Dataset Overview
Shape: 1470 rows × 35 columns

Attrition breakdown:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Missing values: 0 (none expected in this dataset)


In [26]:
# These columns carry no analytical value — they are constants across all rows.
cols_to_drop = ["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"]
df.drop(columns=cols_to_drop, inplace=True)
 
# Encode the target variable as a binary integer (1 = left, 0 = stayed).
# This is needed for correlation and modelling later.
df["AttritionBinary"] = (df["Attrition"] == "Yes").astype(int)

print(f"\nAttrition rate: {df['AttritionBinary'].mean():.1%}")


Attrition rate: 16.1%


In [ ]:
# SECTION 2 — Descriptive Analytics
# =============================================================================
# Descriptive analytics answers: "What is actually in the data?"
# We summarise key recognition and engagement variables before running tests.

In [27]:
print("\n" + "=" * 60)
print("SECTION 2 — Descriptive Analytics")
print("=" * 60)
 
recognition_vars = [
    "PerformanceRating", "PercentSalaryHike",
    "YearsSinceLastPromotion", "TrainingTimesLastYear",
    "StockOptionLevel", "JobLevel"
]
 
engagement_vars = [
    "JobSatisfaction", "WorkLifeBalance",
    "JobInvolvement", "EnvironmentSatisfaction",
    "RelationshipSatisfaction"
]
 
print("\n--- Recognition Variables ---")
print(df[recognition_vars].describe().round(2))
 
print("\n--- Engagement Variables ---")
print(df[engagement_vars].describe().round(2))


SECTION 2 — Descriptive Analytics

--- Recognition Variables ---
       PerformanceRating  PercentSalaryHike  YearsSinceLastPromotion  \
count            1470.00            1470.00                  1470.00   
mean                3.15              15.21                     2.19   
std                 0.36               3.66                     3.22   
min                 3.00              11.00                     0.00   
25%                 3.00              12.00                     0.00   
50%                 3.00              14.00                     1.00   
75%                 3.00              18.00                     3.00   
max                 4.00              25.00                    15.00   

       TrainingTimesLastYear  StockOptionLevel  JobLevel  
count                1470.00           1470.00   1470.00  
mean                    2.80              0.79      2.06  
std                     1.29              0.85      1.11  
min                     0.00              0.00   

In [30]:
# ---- 2A: Average satisfaction by performance rating -------------------------
# PerformanceRating = 3 (Excellent) or 4 (Outstanding) in this dataset.
# Checking whether higher-rated employees report better satisfaction.

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Average Engagement Scores by Performance Rating", fontsize=13, fontweight="bold")
 
for ax, var in zip(axes, ["JobSatisfaction", "WorkLifeBalance", "JobInvolvement"]):
    means = df.groupby("PerformanceRating")[var].mean()
    ax.bar(means.index.astype(str), means.values, color=sns.color_palette(PALETTE)[:len(means)])
    ax.set_title(var)
    ax.set_xlabel("Performance Rating")
    ax.set_ylabel("Mean Score")
    ax.set_ylim(0, 4.5)
 
plt.tight_layout()
plt.savefig("fig1_satisfaction_by_rating.png", dpi=150)
plt.close()
print("\nSaved → fig1_satisfaction_by_rating.png")


Saved → fig1_satisfaction_by_rating.png


In [31]:
# ---- 2B: Attrition by department --------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))
dept_attr = df.groupby("Department")["AttritionBinary"].mean().sort_values(ascending=False) * 100
bars = ax.bar(dept_attr.index, dept_attr.values, color=sns.color_palette(PALETTE)[:len(dept_attr)])
ax.set_title("Attrition Rate by Department (%)", fontweight="bold")
ax.set_ylabel("Attrition Rate (%)")
ax.set_xlabel("Department")
for bar, val in zip(bars, dept_attr.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, f"{val:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("fig2_attrition_by_dept.png", dpi=150)
plt.close()
print("Saved → fig2_attrition_by_dept.png")

Saved → fig2_attrition_by_dept.png


In [32]:
# ---- 2C: Distribution of salary hike by performance rating ------------------
fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column="PercentSalaryHike", by="PerformanceRating", ax=ax, patch_artist=True)
ax.set_title("Salary Hike Distribution by Performance Rating", fontweight="bold")
ax.set_xlabel("Performance Rating")
ax.set_ylabel("% Salary Hike")
plt.suptitle("")
plt.tight_layout()
plt.savefig("fig3_salaryhike_by_rating.png", dpi=150)
plt.close()
print("Saved → fig3_salaryhike_by_rating.png")

Saved → fig3_salaryhike_by_rating.png


In [33]:
# ---- 2D: Cross-tabulation — recognition level vs satisfaction ---------------
# JobSatisfaction and PerformanceRating are both ordinal.
# A cross-tab shows how they co-vary in frequency counts.
crosstab = pd.crosstab(df["PerformanceRating"], df["JobSatisfaction"])
print("\nCross-tabulation — PerformanceRating vs JobSatisfaction")
print(crosstab)


Cross-tabulation — PerformanceRating vs JobSatisfaction
JobSatisfaction      1    2    3    4
PerformanceRating                    
3                  241  237  386  380
4                   48   43   56   79


In [34]:
# ---- 2E: Heatmap of correlations among key variables -----------------------
key_cols = recognition_vars + engagement_vars + ["AttritionBinary"]
corr_matrix = df[key_cols].corr()
 
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, ax=ax, linewidths=0.5
)
ax.set_title("Correlation Heatmap — Recognition & Engagement Variables", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("fig4_correlation_heatmap.png", dpi=150)
plt.close()
print("Saved → fig4_correlation_heatmap.png")

Saved → fig4_correlation_heatmap.png
